In [ ]:
import pandas as pd
import numpy as np
import joblib
import re

from scipy.sparse import csr_matrix, hstack
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [ ]:
best_model = joblib.load("best_model.pkl")
tfidf = joblib.load("tfidf_vectorizer.pkl")

In [ ]:
def extract_features(text):

    words = text.split()

    char_count = len(text)

    word_count = len(words)

    sentence_count = max(
        len(re.findall(r"[.!?]", text)),
        1
    )

    avg_word_length = np.mean(
        [len(i) for i in words]
    ) if words else 0

    digit_ratio = sum(
        c.isdigit()
        for c in text
    ) / max(len(text),1)

    special_ratio = len(
        re.findall(
            r"[^a-zA-Z0-9\s]",
            text
        )
    ) / max(len(text),1)

    exclamation_count = text.count("!")

    question_count = text.count("?")

    has_email = int(
        bool(
            re.search(
                r"\S+@\S+",
                text
            )
        )
    )

    gmail_email = int(
        "@gmail" in text.lower()
    )

    yahoo_email = int(
        "@yahoo" in text.lower()
    )

    outlook_email = int(
        "@outlook" in text.lower()
    )

    salary_present = int(
        bool(
            re.search(
                r"\d",
                text
            )
        )
    )

    salary_daily = int(
        "day" in text.lower()
    )

    salary_monthly = int(
        "month" in text.lower()
    )

    salary_hourly = int(
        "hour" in text.lower()
    )

    avg_sentence_length = min(
        word_count/sentence_count,
        100
    )

    unique_word_ratio = len(
        set(words)
    ) / max(word_count,1)

    repetition_score = (
        1-unique_word_ratio
    )

    long_word_ratio = sum(
        len(i)>8
        for i in words
    ) / max(word_count,1)

    numeric_token_ratio = sum(
        any(c.isdigit() for c in i)
        for i in words
    ) / max(word_count,1)

    return [

        char_count,

        word_count,

        sentence_count,

        avg_word_length,

        digit_ratio,

        special_ratio,

        exclamation_count,

        question_count,

        has_email,

        gmail_email,

        yahoo_email,

        outlook_email,

        salary_present,

        salary_daily,

        salary_monthly,

        salary_hourly,

        avg_sentence_length,

        unique_word_ratio,

        repetition_score,

        long_word_ratio,

        numeric_token_ratio

    ]

In [ ]:
import pandas as pd

external_df = pd.read_excel("external_test.xlsx")
external_df.head()


In [ ]:
external_df["combined_text"] = (

    external_df["job_title"].fillna("")

    + " "

    + external_df["company_name"].fillna("")

    + " "

    + external_df["job_description"].fillna("")

)

In [ ]:
X_text = tfidf.transform(

    external_df["combined_text"]

)

In [ ]:
manual = external_df["combined_text"].apply(

    extract_features

)

manual = pd.DataFrame(

    manual.tolist(),

    columns=[

        "char_count",
        "word_count",
        "sentence_count",
        "avg_word_length",
        "digit_ratio",
        "special_ratio",
        "exclamation_count",
        "question_count",

        "has_email",
        "gmail_email",
        "yahoo_email",
        "outlook_email",

        "salary_present",
        "salary_daily",
        "salary_monthly",
        "salary_hourly",

        "avg_sentence_length",
        "unique_word_ratio",
        "repetition_score",
        "long_word_ratio",
        "numeric_token_ratio"

    ]

)

X_manual = csr_matrix(

    manual.values

)

In [ ]:
X = hstack([

    X_text,

    X_manual

])

In [ ]:
X = hstack([

    X_text,

    X_manual

])

In [ ]:
pred = best_model.predict(X)

prob = best_model.predict_proba(X)[:,1]

external_df["Prediction"] = pred

external_df["Probability"] = prob

In [ ]:
print(

classification_report(

external_df["label"],

pred

)

)

print()

print(

confusion_matrix(

external_df["label"],

pred

)

)

print()

print(

"Accuracy :",

accuracy_score(

external_df["label"],

pred

)

)

print(

"Precision :",

precision_score(

external_df["label"],

pred

)

)

print(

"Recall :",

recall_score(

external_df["label"],

pred

)

)

print(

"F1 :",

f1_score(

external_df["label"],

pred

)

)

In [ ]:
wrong = external_df[

external_df["label"]

!=

external_df["Prediction"]

]

print(

wrong.shape

)

wrong.head(20)

In [ ]:
print(
classification_report(
external_df["label"],
pred
)
)

In [ ]:
print(wrong.shape)

wrong[
[
"job_title",
"company_name",
"label",
"Prediction",
"Probability"
]
]

In [ ]:
print(
confusion_matrix(
external_df["label"],
pred
)
)

In [ ]:
external_df[
[
"job_title",
"Prediction",
"Probability"
]
].head(20)

In [ ]:
print(
external_df["Probability"].describe()
)

In [ ]:
fake_pred = external_df[
external_df["Prediction"]==1
]

real_pred = external_df[
external_df["Prediction"]==0
]

print(fake_pred.shape)
print(real_pred.shape)